# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstitute_hdf5_from_s3.py` but uses the **local filesystem** for Parquet storage and **SQLite** for metadata. No cloud credentials needed.

**Core workflow**
1. Ingest a **GMI** and an **SSMIS** granule → Parquet partitions on local disk + SQLite metadata (two instruments so the overlap analytics in step 10 have something to compare)
2. Find intersecting data for a bounding box via STARE SIDs + SQLite (level 4)
3. Load intersecting Parquet partitions from disk
4. Reconstitute an HDF5 file (both S1 and S2 scans) from the level-4 Parquet partitions
5. Compare the reconstituted structure with the original granule
6. Verify SQLite metadata

**Temporal features** (temporal-stare-pods issues 01–06)
7. Temporal catalog — every chunk carries `[t_start, t_end]` + podcode
8. Period-filtered intersection — data-level `[t_start, t_end]` overlap
9. VCF temporal roll-up — union range per pod, on the fly
10. Multi-instrument overlap analytics — the slide-8/9 rendezvous views


In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
#                       "-q"])

In [2]:
import os
import sqlite3
import h5py
from starepandas.demo_lib import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [3]:
# Parquet store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
import starepandas
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5",
    ),
)

# A second instrument (SSMIS) so the overlap analytics (step 10) span two
# instruments. The F18 granule (2025-01-05) is the closest in time to the GMI
# granule (2025-01-01) among the in-repo samples.
SSMIS_GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE_SSMIS",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5",
    ),
)

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/reconsitution/gmi_local_reconstituted.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstituted HDF5 (e.g. 3× the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"SSMIS      : {os.path.basename(SSMIS_GRANULE_FILE)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"STARE level: {STARE_LEVEL}")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

Granule    : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
SSMIS      : 1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5
Datasets   : ['GMI_S1', 'GMI_S2']
BBox       : None  (None = full granule)
STARE level: 4
Local root : /tmp/stare_pods_local
Clean first: True


## Step 1 — Ingest granule → local Parquet + SQLite

In [4]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

Removed existing data at /tmp/stare_pods_local


In [5]:
%%time
import time
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=STARE_LEVEL)
print(f"GMI  : written {len(local_paths)} scan path(s).")

ssmis_paths = demo.ingest_granules(SSMIS_GRANULE_FILE, instrument='SSMIS', level=STARE_LEVEL)
print(f"SSMIS: written {len(ssmis_paths)} scan path(s).")


INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5 (granule=1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


INFO:starepandas.ingest:Found 1 SSMIS file(s)


INFO:starepandas.ingest:Processing 1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5


GMI  : written 2 scan path(s).


INFO:starepandas.ingest:✓ Stored 1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B.HDF5 (granule=1C.F18.SSMIS.XCAL2021-V.20250105-S222535-E000725.078504.V07B) → /tmp/stare_pods_local


INFO:starepandas.ingest:Ingested 4 Parquet dataset(s)


SSMIS: written 4 scan path(s).
CPU times: user 18.6 s, sys: 1.19 s, total: 19.8 s
Wall time: 19.8 s


## Step 2 — Find intersecting data via STARE SIDs

In [6]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all partitions will be loaded (full granule reconstitution)")

intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'])
print(f"Found {len(intersecting)} metadata row(s).")
intersecting[['Dataset', 'grouped_id', 'group_path']]

INFO:starepandas.demo_lib:Loaded all 514 partitions for GMI


No bbox filter — all partitions will be loaded (full granule reconstitution)
Found 514 metadata row(s).


,Dataset,grouped_id,group_path
0,GMI_S1,2096425626540965892,/tmp/stare_pods_local/q32/q322/q3220/q32203/q3...
1,GMI_S1,2094173826727280644,/tmp/stare_pods_local/q32/q322/q3220/q32202/q3...
2,GMI_S1,2118943624677818372,/tmp/stare_pods_local/q32/q322/q3223/q32231/q3...
3,GMI_S1,2109936425423077380,/tmp/stare_pods_local/q32/q322/q3222/q32221/q3...
4,GMI_S1,2175238620019949572,/tmp/stare_pods_local/q33/q330/q3301/q33012/q3...
...,...,...,...
509,GMI_S2,2107684625609392132,/tmp/stare_pods_local/q32/q322/q3222/q32220/q3...
510,GMI_S2,2127950823932559364,/tmp/stare_pods_local/q32/q323/q3230/q32301/q3...
511,GMI_S2,2112188225236762628,/tmp/stare_pods_local/q32/q322/q3222/q32222/q3...
512,GMI_S2,2114440025050447876,/tmp/stare_pods_local/q32/q322/q3222/q32223/q3...


## Step 3 — Load intersecting Parquet partitions from disk

In [7]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions found.")
    data_dict = {}

INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S1


INFO:starepandas.demo_lib:✓ Combined 659243 rows for GMI_S2


GMI_S1: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Tc5,Tc6,...,incidenceAngleIndex5,incidenceAngleIndex6,incidenceAngleIndex7,incidenceAngleIndex8,incidenceAngleIndex9,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.416740,40.916428,2096919498826664619,2025-01-01 03:43:47.516,162.190002,88.510002,183.320007,116.150002,206.300003,209.000000,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075
1,-60.429089,40.802353,2096462762016624139,2025-01-01 03:43:47.516,162.830002,89.080002,183.410004,116.639999,206.479996,209.020004,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075
2,-60.442020,40.688499,2096462641002782731,2025-01-01 03:43:47.516,161.820007,88.800003,183.320007,117.000000,206.729996,209.750000,...,1,1,1,1,1,180,-65.10363,43.338219,451.328461,61567.000075


GMI_S2: 659243 rows, columns: ['lat', 'lon', 'sids', 'timestamp', 'Tc1', 'Tc2'] …


,lat,lon,sids,timestamp,Tc1,Tc2,Tc3,Tc4,Quality,incidenceAngle,...,sunLocalTime,incidenceAngleIndex1,incidenceAngleIndex2,incidenceAngleIndex3,incidenceAngleIndex4,SCstatus_SCorientation,SCstatus_SClatitude,SCstatus_SClongitude,SCstatus_SCaltitude,SCstatus_FractionalGranuleNumber
0,-60.956139,41.155605,2095643662640840171,2025-01-01 03:43:47.516,256.399994,251.600006,248.050003,255.509995,0,49.57,...,6.415025,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075
1,-60.967239,41.053024,2095644871184156395,2025-01-01 03:43:47.516,256.359985,250.559998,248.229996,254.919998,0,49.57,...,6.408187,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075
2,-60.978867,40.950649,2095644225044829739,2025-01-01 03:43:47.516,255.940002,250.279999,247.250000,254.279999,0,49.57,...,6.401363,1,1,1,1,180,-65.10363,43.339439,451.328461,61567.000075


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [8]:
%%time
import time
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]

recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    granule_name=granule_basename,
)
print(f"Written to: {recon_path}")

INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over bbox=None


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over bbox=None


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/reconsitution/gmi_local_reconstituted.h5


Written to: /tmp/reconsitution/gmi_local_reconstituted.h5
CPU times: user 2.04 s, sys: 441 ms, total: 2.49 s
Wall time: 1.87 s


## Step 5 — Structure comparison: reconstituted vs original

In [9]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_local_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear    

## Step 6 — SQLite metadata verification

In [10]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} partition(s)")

SQLite DB: /tmp/stare_pods_local/metadata.db
  GMI_S1: 263 partition(s)
  GMI_S2: 251 partition(s)
  SSMIS_S1: 380 partition(s)
  SSMIS_S2: 380 partition(s)
  SSMIS_S3: 378 partition(s)
  SSMIS_S4: 380 partition(s)


## Step 7 — Temporal catalog: every chunk carries `[t_start, t_end]` + podcode

Each ingested chunk now records its temporal range and quaternary pod code. `load_local_temporal_catalog` returns the thin projection the analytics use (`podcode / Dataset / t_start / t_end`) — never the heavy `MetadataJson`.

In [11]:
from starepandas.io.granules import load_local_temporal_catalog, load_local_vcf
from starepandas.overlap import (
    rendezvous_events, overlap_matrix, overlap_pod_table,
    pair_drilldown, pod_drilldown,
)

catalog = load_local_temporal_catalog(demo.db_path)
print(f"Thin catalog: {len(catalog)} chunks across {catalog['Dataset'].nunique()} datasets")
display(catalog.groupby('Dataset').agg(
    chunks=('podcode', 'size'),
    first_start=('t_start', 'min'),
    last_end=('t_end', 'max'),
))
catalog.head(6)

Thin catalog: 2032 chunks across 6 datasets


,chunks,first_start,last_end
Dataset,,,
GMI_S1,263,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739
GMI_S2,251,2025-01-01 03:43:47.516,2025-01-01 05:16:58.739
SSMIS_S1,380,2025-01-05 22:25:35.236,2025-01-06 00:07:25.237
SSMIS_S2,380,2025-01-05 22:25:35.236,2025-01-06 00:07:25.237
SSMIS_S3,378,2025-01-05 22:25:35.236,2025-01-06 00:07:25.237
SSMIS_S4,380,2025-01-05 22:25:35.236,2025-01-06 00:07:25.237


,podcode,Dataset,t_start,t_end
0,q32221,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:49.391
1,q32231,GMI_S2,2025-01-01 03:43:47.516,2025-01-01 03:43:49.391
2,q33012,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:49.391
3,q32231,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:43:58.766
4,q33032,GMI_S2,2025-01-01 03:43:47.516,2025-01-01 03:44:23.140
5,q33032,GMI_S1,2025-01-01 03:43:47.516,2025-01-01 03:44:40.015


## Step 8 — Period-filtered intersection

`find_intersecting_data(..., period=(start, end))` keeps only chunks whose **data-level** range `[t_start, t_end]` overlaps the period — ANDed with the spatial pod match. A window bracketing the GMI pass returns its chunks; a window days away returns none. (This is distinct from the granule-level `start_date`/`end_date` filename filter.)

In [12]:
import pandas as pd

gmi = catalog[catalog['Dataset'].str.startswith('GMI')]
gmi_start, gmi_end = gmi['t_start'].min(), gmi['t_end'].max()
match_period = (gmi_start - pd.Timedelta(hours=1), gmi_end + pd.Timedelta(hours=1))
miss_period  = (gmi_start - pd.Timedelta(days=10), gmi_start - pd.Timedelta(days=9))

hit  = demo.find_intersecting_data(None, ['GMI'], period=match_period)
miss = demo.find_intersecting_data(None, ['GMI'], period=miss_period)

print(f"GMI pass window   : [{gmi_start}, {gmi_end}]")
print(f"bracketing period -> {len(hit)} chunks")
print(f"9-10 days earlier -> {len(miss)} chunks")

INFO:starepandas.demo_lib:Loaded all 514 partitions for GMI


GMI pass window   : [2025-01-01 03:43:47.516000, 2025-01-01 05:16:58.739000]
bracketing period -> 514 chunks
9-10 days earlier -> 0 chunks


## Step 9 — VCF temporal roll-up

The temporal hierarchy ("Virtual Collection File") is queryable on the fly: `load_local_vcf(db, level)` groups chunks by their level-`level` ancestor pod and returns each pod's union range `[min(t_start), max(t_end)]` plus its child count. Nothing is materialized — a different level just re-groups the same thin load.

In [13]:
vcf = load_local_vcf(demo.db_path, level=1)
print(f"{len(vcf)} level-1 VCF nodes (one per octant subtree)")
vcf

26 level-1 VCF nodes (one per octant subtree)


,podcode,t_start,t_end,n_chunks,n_without_range
0,q00,2025-01-05 23:48:16.528,2025-01-05 23:57:30.947,76,0
1,q01,2025-01-01 03:46:47.515,2025-01-01 04:01:04.385,64,0
2,q02,2025-01-05 23:31:45.409,2025-01-05 23:42:06.282,140,0
3,q03,2025-01-05 23:40:27.551,2025-01-05 23:57:15.758,140,0
4,q10,2025-01-05 23:31:54.902,2025-01-05 23:32:04.397,4,0
5,q11,2025-01-01 03:59:17.511,2025-01-01 04:05:40.009,33,0
6,q12,2025-01-01 04:16:45.630,2025-01-01 04:21:19.379,18,0
7,q13,2025-01-01 04:04:11.884,2025-01-01 04:18:34.380,56,0
8,q22,2025-01-05 22:40:12.434,2025-01-05 22:43:09.013,16,0
9,q30,2025-01-01 05:06:23.117,2025-01-05 22:40:27.622,158,0


## Step 10 — Multi-instrument overlap analytics (slides 8/9)

`rendezvous_events` sweeps the catalog for passes simultaneously present in a pod within Δt; the matrix / pod-table / drill-downs all aggregate that one events frame.

> **Note on the sample data.** The in-repo GMI (2025-01-01) and SSMIS (2025-01-05) passes are days apart and land in different pods, so a realistic Δt finds **no** rendezvous on them — the sweep is correct, the sample granules simply don't co-locate. The small synthetic catalog below stands in to show what the views look like when data *does* overlap.

In [14]:
dt = pd.Timedelta(minutes=30)
real_events = rendezvous_events(catalog, dt)
print(f"Rendezvous over the ingested GMI+SSMIS catalog (dt={dt}): "
      f"{len(real_events)} events")

Rendezvous over the ingested GMI+SSMIS catalog (dt=0 days 00:30:00): 0 events


### Illustrative synthetic catalog (co-located passes)

Three pods: one where GMI + SSMIS + ATMS all pass within 30 min (a **trio**), one with a GMI + SSMIS **pair**, and one with GMI alone (no rendezvous). Same shape a temporal-catalog loader returns.

In [15]:
m = lambda mins: pd.Timedelta(minutes=mins)
t = pd.Timestamp('2025-01-01 10:00')
demo_cat = pd.DataFrame(
    [   # pod q13011: GMI + SSMIS + ATMS within 30 min -> a trio
        ('q13011', 'GMI_S1',   t,         t + m(2)),
        ('q13011', 'SSMIS_S1', t + m(12), t + m(15)),
        ('q13011', 'ATMS_S1',  t + m(20), t + m(23)),
        # pod q13012: GMI + SSMIS -> a pair
        ('q13012', 'GMI_S1',   t + m(120), t + m(123)),
        ('q13012', 'SSMIS_S1', t + m(140), t + m(144)),
        # pod q13013: GMI alone -> no rendezvous
        ('q13013', 'GMI_S1',   t + m(300), t + m(303)),
    ],
    columns=['podcode', 'Dataset', 't_start', 't_end'],
)
ev = rendezvous_events(demo_cat, dt)
print(f"{len(ev)} events over {demo_cat['podcode'].nunique()} pods")

print('\nInstrument x instrument matrix — pods where A & B rendezvous (slide 8):')
display(overlap_matrix(ev))

print('Per-pod n-way combination counts (slide 9):')
display(overlap_pod_table(ev))

print('GMI-SSMIS pair drill-down (pods + times):')
display(pair_drilldown(ev, 'GMI', 'SSMIS'))

print("Subtree drill-down under 'q1301' (rolls up q13011/12/13):")
display(pod_drilldown(ev, 'q1301'))

3 events over 3 pods

Instrument x instrument matrix — pods where A & B rendezvous (slide 8):


,ATMS,GMI,SSMIS
ATMS,0,1,1
GMI,1,0,2
SSMIS,1,2,0


Per-pod n-way combination counts (slide 9):


n_instruments,2,3
podcode,,
q13011,3,1
q13012,1,0


GMI-SSMIS pair drill-down (pods + times):


,podcode,frequency,times
0,q13011,1,[2025-01-01 10:12:00]
1,q13012,1,[2025-01-01 12:20:00]


Subtree drill-down under 'q1301' (rolls up q13011/12/13):


,instruments,n_instruments,frequency,times
0,"(ATMS, GMI)",2,1,[2025-01-01 10:20:00]
1,"(ATMS, SSMIS)",2,1,[2025-01-01 10:20:00]
2,"(GMI, SSMIS)",2,2,"[2025-01-01 10:12:00, 2025-01-01 12:20:00]"
3,"(ATMS, GMI, SSMIS)",3,1,[2025-01-01 10:20:00]
